# 1D firing rate maps (per-trajectory, not pooled)

1D counterpart of `sindhu_firing_rate_maps_clean.ipynb`. Computes occupancy and firing-rate maps for one neuron at a time, separately for each trajectory in `trajectory_results.pkl`.

Fixes vs. `1D-place-cell-classifier.ipynb`:
- Occupancy map and firing-rate map use the *same* `np.histogram(..., range=)` call signature, with an explicit, shared `bounds` computed once from all trajectories — not each call's own data min/max — so every trajectory's map lands on an identical grid.
- Activation values are converted to amplitude (`np.abs`) before binning, instead of thresholding/comparing raw complex activity (which silently discards the imaginary part — this is what caused the `ComplexWarning` in the original notebook).
- No arbitrary `[1, 100]` rescaling of the output — the returned map is the real occupancy-normalized rate, in the activation's own units.
- No dependence on module-level globals (`lim`, `reso`) — every parameter is passed explicitly.
- Scores (sparsity, spatial information, place-cell identification) are intentionally left out, same scope as the 2D clean notebook — this is for looking at the raw firing-rate maps only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.ndimage import gaussian_filter1d

In [ ]:
# Map resolution. Tune N_BINS against your actual per-trajectory sample count once loaded below —
# aim for roughly 1 raw sample per bin before smoothing (same reasoning as the 2D notebook).
N_BINS = 40
SIGMA = 1.0     # Gaussian smoothing bandwidth, in bins
EPS = 1e-10

### Input

In [ ]:
def get_input_data(trajectory_results_path, layer_name):
    '''
    Expects trajectory_results_path to unpickle to a list of dicts, each with a
    'pred_position' array and a layer_name array (as in trajectory_results.pkl).
    Trajectories may have different lengths, so positions/activations are returned
    as lists (one entry per trajectory), not stacked arrays.
    Returns positions_list (each (T,)) and activations_list (each (T, n_neurons)).
    '''
    with open(trajectory_results_path, 'rb') as f:
        trajectory_results = pickle.load(f)

    positions_list = []
    activations_list = []
    for traj in trajectory_results:
        pos = np.asarray(traj['pred_position']).reshape(-1)
        act = np.asarray(traj[layer_name]).reshape(-1, np.asarray(traj[layer_name]).shape[-1])
        positions_list.append(pos)
        activations_list.append(act)

    return positions_list, activations_list

In [ ]:
def get_track_bounds(positions_list):
    '''Fixed (min, max) computed once across every trajectory, shared by every map so bins line up.'''
    all_pos = np.concatenate(positions_list)
    return (float(all_pos.min()), float(all_pos.max()))

### Mapping functions

In [ ]:
def occupancy_map_func(positions, bins=N_BINS, bounds=None, sigma=SIGMA):
    '''
    positions: (T,) single trajectory.
    Returns a smoothed occupancy-count map, shape (bins,), and the bin edges.
    '''
    H, edges = np.histogram(positions, bins=bins, range=bounds)
    occupancy = gaussian_filter1d(H.astype(float), sigma=sigma)
    return occupancy, edges

In [ ]:
def firing_rate_map_func(positions, activity, occupancy_map, bins=N_BINS, bounds=None, sigma=SIGMA, eps=EPS):
    '''
    positions: (T,) single trajectory.
    activity:  (T,) one neuron's activation for that same trajectory (real or complex).
    occupancy_map: output of occupancy_map_func for this same trajectory (same bins/bounds/sigma).
    Returns the occupancy-normalized firing-rate map, shape (bins,), on the same grid as occupancy_map
    since both come from the same histogram(bins=, range=) call.
    '''
    amplitude = np.abs(activity)

    activity_sum, _ = np.histogram(positions, bins=bins, range=bounds, weights=amplitude)
    activity_sum = gaussian_filter1d(activity_sum, sigma=sigma)

    rate_map = activity_sum / (occupancy_map + eps)
    return rate_map

In [ ]:
def firing_rate_maps_for_neuron(positions_list, activations_list, neuron_idx, bins=N_BINS, bounds=None, sigma=SIGMA, eps=EPS):
    '''
    positions_list:   list of (T,) arrays, one per trajectory.
    activations_list: list of (T, n_neurons) arrays, one per trajectory.
    Computes one occupancy map + one firing-rate map per trajectory (not pooled).
    Returns firing_maps, occupancy_maps, each shape (n_traj, bins).
    '''
    if bounds is None:
        bounds = get_track_bounds(positions_list)

    n_traj = len(positions_list)
    firing_maps = np.zeros((n_traj, bins))
    occupancy_maps = np.zeros((n_traj, bins))

    for t in range(n_traj):
        occ_map, _ = occupancy_map_func(positions_list[t], bins=bins, bounds=bounds, sigma=sigma)
        fr_map = firing_rate_map_func(
            positions_list[t], activations_list[t][:, neuron_idx], occ_map,
            bins=bins, bounds=bounds, sigma=sigma, eps=eps
        )
        occupancy_maps[t] = occ_map
        firing_maps[t] = fr_map

    return firing_maps, occupancy_maps, bounds

### Plotting

In [ ]:
def plot_random_trajectory_maps(firing_maps, neuron_idx, bounds, n_show=6, seed=None):
    '''
    Plots the firing-rate map for n_show randomly chosen trajectories, for one neuron.
    firing_maps: (n_traj, bins) as returned by firing_rate_maps_for_neuron.
    '''
    rng = np.random.default_rng(seed)
    n_traj, bins = firing_maps.shape
    n_show = min(n_show, n_traj)
    chosen = rng.choice(n_traj, size=n_show, replace=False)
    bin_centers = np.linspace(bounds[0], bounds[1], bins)

    ncols = min(3, n_show)
    nrows = int(np.ceil(n_show / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False)

    for ax, traj_idx in zip(axes.ravel(), chosen):
        ax.plot(bin_centers, firing_maps[traj_idx], linewidth=2)
        ax.set_title(f'Trial {traj_idx}')
        ax.set_xlabel('Position')
        ax.set_ylabel('Firing rate')

    for ax in axes.ravel()[len(chosen):]:
        ax.axis('off')

    fig.suptitle(f'Neuron {neuron_idx} — firing rate maps ({n_show} random trials)')
    plt.tight_layout()
    plt.show()

### Example usage

In [ ]:
layer_name = "d3"
trajectory_results_path = "trajectory_results.pkl"

positions_list, activations_list = get_input_data(trajectory_results_path, layer_name)

print('num trajectories:', len(positions_list))
print('per-trajectory lengths (first 5):', [len(p) for p in positions_list[:5]])
print('num neurons:', activations_list[0].shape[1])

In [ ]:
neuron_idx = 0

firing_maps, occupancy_maps, bounds = firing_rate_maps_for_neuron(positions_list, activations_list, neuron_idx)
print('track bounds used for every trajectory:', bounds)

plot_random_trajectory_maps(firing_maps, neuron_idx, bounds, n_show=6, seed=0)